Link: [link](https://www.kaggle.com/competitions/london-house-price-prediction-advanced-techniques/data)

# **1. Import libraries**

In [ ]:
!pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, Normalizer, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.pipeline import Pipeline
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import mean_absolute_error, r2_score

# 2. **Load dataset**

In [ ]:
train_df = pd.read_csv('/kaggle/input/london-house-price-prediction-advanced-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/london-house-price-prediction-advanced-techniques/test.csv')

In [ ]:
combine = [train_df, test_df]

In [ ]:
for data in combine:
    data['time'] = pd.to_datetime(dict(year=data['sale_year'], month=data['sale_month'], day = 15))
    data['time'] = data['time'].dt.to_period('M')
    data['time_numeric'] = (data['time'].dt.to_timestamp() - data['time'].min().to_timestamp()) / np.timedelta64(1, 'D')

In [ ]:
train_df = train_df.set_index('time')
test_df = test_df.set_index('time')
combine = [train_df, test_df]

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
print('Features: \n', train_df.columns)

## Preliminary Information

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

In [ ]:
test_df.info()

In [ ]:
test_df.describe()

In [ ]:
train_df['price'].head()

# **3. EDA**

In [ ]:
num_features = ['latitude',	'longitude',	'bathrooms',	'bedrooms', \
                                  'floorAreaSqM',	'livingRooms', 'sale_month', 'sale_year', 'price']

## 3.1. Identify columns that contain NULL values

In [ ]:
df = train_df.isnull().sum() / len(train_df) * 100
df[df > 0]

In [ ]:
df = test_df.isnull().sum()
df[df > 0]

## 3.2. Identify outlier values

In [ ]:
g, ax = plt.subplots(3, 3, figsize=(15, 25))
ax = ax.flatten()
for i, col in enumerate(train_df[num_features].columns):
    sns.boxplot(data = train_df, x = col, ax=ax[i])

## 3.3. univariate analysis

In [ ]:
train_df.nunique()

In [ ]:
sns.countplot(data = train_df, x = 'tenure')

In [ ]:
train_df['outcode'].value_counts()

In [ ]:
g, ax = plt.subplots(3, 3, figsize=(19, 20))
ax = ax.flatten()
for i, col in enumerate(train_df[num_features].columns):
    sns.histplot(data = train_df.dropna(subset = [col]), x = col, bins = 40, ax=ax[i], kde = True)
    ax[i].set_title(col)

In [ ]:
train_df['price']

In [ ]:
monthly_avg = train_df.groupby(train_df.index)['price'].mean()
moving_average = monthly_avg.rolling( \
    window = 12, \
    center = True, \
).mean()

moving_average.plot(linewidth=3, legend = False)

## 3.4. Multivariate analysis

### Feature - feature

In [ ]:
train_df.groupby(['livingRooms', 'bedrooms'])['ID'].count()

In [ ]:
df = train_df.copy()
df['latitude_bin'] = pd.cut(df['latitude'], bins=10)

area_by_lat_bin = df.groupby('latitude_bin')['floorAreaSqM'].mean()

area_by_lat_bin.plot(kind='bar', figsize=(12, 5), color='skyblue')

In [ ]:
df = train_df.copy()
df['longitude_bin'] = pd.cut(df['longitude'], bins=10)

area_by_long_bin = df.groupby('longitude_bin')['floorAreaSqM'].mean()

area_by_long_bin.plot(kind='bar', figsize=(12, 5), color='skyblue')

In [ ]:
avg_price_per_month = train_df.groupby(['sale_month', 'sale_year'])['price'].mean().unstack()

avg_price_per_month.loc[:, avg_price_per_month.columns <= 2005].plot()

In [ ]:
avg_price_per_month = train_df.groupby(['sale_month', 'sale_year'])['price'].mean().unstack()

avg_price_per_month.loc[:, (avg_price_per_month.columns > 2005) & (avg_price_per_month.columns <= 2015)].plot()

In [ ]:
avg_price_per_month = train_df.groupby(['sale_month', 'sale_year'])['price'].mean().unstack()

avg_price_per_month.loc[:, (avg_price_per_month.columns > 2015)].plot()

### Feature - output

In [ ]:
train_df.groupby('tenure')['price'].mean().plot(kind='bar')

In [ ]:
train_df.groupby('outcode')['price'].mean()

In [ ]:
g, ax = plt.subplots(3, 3, figsize=(15, 15))
ax = ax.flatten()
for i, col in enumerate(train_df[num_features].columns):
    sns.scatterplot(data=train_df, x=col, y='price', ax = ax[i])

### Correlation Matrix

In [ ]:
sns.heatmap(train_df[num_features].corr(),  cmap = 'coolwarm', annot=True)

## 3.5. Insights from Data Exploration
1. The country column can be dropped as it contains only one unique value.

2. Tenure values Feudal and Shared are rare.

3. Missing data in features: bathrooms, bedrooms, floorAreaSqM, livingRooms, tenure, propertyType, currentEnergyRating.

4. Possible outliers in: bathrooms, bedrooms, floorAreaSqM, livingRooms.

5. Price is mostly concentrated between 0 and 10,000,000.

6. Bathrooms and floorAreaSqM are moderately correlated with price.

7. Bathrooms, bedrooms, and floorAreaSqM are highly correlated with each other.

8. Latitude and Longitude show high-priced properties clustered in specific areas.

9. FloorAreaSqM shows a linear trend, unlike bathrooms, bedrooms, and livingRooms.

# **4. Wrangle data**

In [ ]:
test_df['price'] = np.nan

## 4.1. Completing and Creating

### fullAddress

In [ ]:
for data in combine:
    data['street'] = data['fullAddress'].apply(lambda z : ' '.join(z.split(',')[-3].split(' ')[-2:]))

In [ ]:
train_df[['street']]

In [ ]:
len(train_df['street'].unique())

In [ ]:
len(set(train_df['street']) | set(test_df['street']))

### postcode

In [ ]:
for data in combine:
    data['postcode'] = data['postcode'].apply(lambda x : x.split(' ')[1])

### country

In [ ]:
for data in combine:
    data.drop('country', axis = 1, inplace = True)

### latitude & longitude

Do nothing

### livingRooms

In [ ]:
si = SimpleImputer(strategy = 'most_frequent')
si.fit(train_df[['livingRooms']])
for data in combine:
    data['livingRooms'] = si.transform(data[['livingRooms']]).ravel()

### floorAreaSqM

In [ ]:
def get_mode(x):
    return x.mode().iloc[0] if not x.mode().empty else np.nan

In [ ]:
get_mode(train_df['floorAreaSqM'])

In [ ]:
(train_df['floorAreaSqM'] == 55.0).sum()

In [ ]:
df = pd.DataFrame({
    'min': train_df.groupby('outcode')['floorAreaSqM'].min(),
    'max': train_df.groupby('outcode')['floorAreaSqM'].max(),
    'mode': train_df.groupby('outcode')['floorAreaSqM'].apply(get_mode),
    'mean': train_df.groupby('outcode')['floorAreaSqM'].mean(),
    'count': train_df.groupby('outcode')['floorAreaSqM'].count()
})

df

In [ ]:
for data in combine:
    data['have_area'] = data['floorAreaSqM'].notna()

In [ ]:
train_df.groupby('propertyType')['floorAreaSqM'].mean()

In [ ]:
mp = train_df.groupby('propertyType')['floorAreaSqM'].mean()

missing_train = train_df[train_df['floorAreaSqM'].isna()]

missing_test = test_df[test_df['floorAreaSqM'].isna()]

train_df.loc[train_df['floorAreaSqM'].isna(), 'floorAreaSqM'] = missing_train['propertyType'].map(mp).fillna(85.0) 
test_df.loc[test_df['floorAreaSqM'].isna(), 'floorAreaSqM'] = missing_test['propertyType'].map(mp).fillna(85.0) 
combine = [train_df, test_df]

### bathrooms

In [ ]:
mp = train_df.groupby('propertyType')['bathrooms'].mean()

missing_train = train_df[train_df['bathrooms'].isna()]

missing_test = test_df[test_df['bathrooms'].isna()]

train_df.loc[train_df['bathrooms'].isna(), 'bathrooms'] = missing_train['propertyType'].map(mp).fillna(train_df['bathrooms'].median()) 
test_df.loc[test_df['bathrooms'].isna(), 'bathrooms'] = missing_test['propertyType'].map(mp).fillna(train_df['bathrooms'].median()) 
combine = [train_df, test_df]

### bedrooms

In [ ]:
mp = train_df.groupby('propertyType')['bedrooms'].mean()

missing_train = train_df[train_df['bedrooms'].isna()]

missing_test = test_df[test_df['bedrooms'].isna()]

train_df.loc[train_df['bedrooms'].isna(), 'bedrooms'] = missing_train['propertyType'].map(mp).fillna(train_df['bedrooms'].median()) 
test_df.loc[test_df['bedrooms'].isna(), 'bedrooms'] = missing_test['propertyType'].map(mp).fillna(train_df['bathrooms'].median()) 
combine = [train_df, test_df]

### tenure

In [ ]:
si = SimpleImputer(strategy = 'most_frequent')
si.fit(train_df[['tenure']])
for data in combine:
    data['tenure'] = si.transform(data[['tenure']]).ravel()

In [ ]:
for data in combine:
    data['isFreehold'] = data['tenure'] == 'Freehold'

### propertyType

In [ ]:
si = SimpleImputer(strategy = 'most_frequent')
si.fit(train_df[['propertyType']])
for data in combine:
    data['propertyType'] = si.transform(data[['propertyType']]).ravel()

### currentEnergyRating

In [ ]:
si = SimpleImputer(strategy = 'most_frequent')
si.fit(train_df[['currentEnergyRating']])
for data in combine:
    data['currentEnergyRating'] = si.transform(data[['currentEnergyRating']]).ravel()

### sale_month & sale_year

In [ ]:
forecast_origin = train_df.index.max()
forecast_lead = test_df.index.min() - forecast_origin
forecast_horizon = test_df.index.max() - test_df.index.min()
# strategy: DirRec

print("Forecast origin:", forecast_origin)
print("Lead time:", forecast_lead.n)
print("Forecast horizon (months):", forecast_horizon.n)

In [ ]:
dp_train = DeterministicProcess(
    index = train_df.index.unique(),
    constant = True,
    order = 4,
    drop = True
)

tmp_train = dp_train.in_sample()

train_df = train_df.join(tmp_train, how = 'left')

In [ ]:
tmp_test = dp_train.out_of_sample(steps = forecast_horizon.n + forecast_lead.n)

test_df = test_df.join(tmp_test, how = 'left')
test_df.index.name = 'time'

In [ ]:
combine = [train_df, test_df]

### bathrooms & bedrooms & livingRooms

In [ ]:
for data in combine:
    data['rooms'] = data['bathrooms'] + 1.5 * data['bedrooms'] + 3 * data['livingRooms']

## 4.2. Encoder

In [ ]:
train_df.isnull().sum()

In [ ]:
test_df.head()

In [ ]:
test_df.isnull().sum()

In [ ]:
class Encoder(BaseEstimator, TransformerMixin):
    '''
    // street, postcode, outcode, tenure, propertyType, currentEnergyRating
    '''
    def __init__(self):
        self.encoder1 = {}
        self.encoder2 = {}
        self.cols = ['street', 'postcode', 'outcode', 'tenure', 'propertyType']

    def fit(self, X, y=None):
        X = X.copy()
        X['price'] = y
        for col in self.cols:
            gm = X.groupby(col)['price'].agg(['median', 'mean', 'count'])
            self.encoder1[f'{col}_median'] = gm['median']
            self.encoder2[f'{col}_median'] = X['price'].median()
            self.encoder1[f'{col}_mean'] = gm['mean']
            self.encoder2[f'{col}_mean'] = X['price'].mean()
        # currentEnergyRating
        custom_order = [['G', 'F', 'E', 'D', 'C', 'B', 'A']]
        lb = OrdinalEncoder(categories=custom_order)
        self.encoder1['currentEnergyRating'] = lb.fit(X[['currentEnergyRating']])
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[f'{col}_median'] = X[col].map(self.encoder1[f'{col}_median'])
            X[f'{col}_median'] = X[f'{col}_median'].fillna(self.encoder2[f'{col}_median'])
            X[f'{col}_mean'] = X[col].map(self.encoder1[f'{col}_mean'])
            X[f'{col}_mean'] = X[f'{col}_mean'].fillna(self.encoder2[f'{col}_mean'])
            X = X.drop(col, axis = 1)
        # currentEnergyRating
        X['currentEnergyRating'] = self.encoder1['currentEnergyRating'].transform(X[['currentEnergyRating']])
        return X

# **5. Chosing feature**

In [ ]:
X1_feature = tmp_train.columns.tolist()
X2_feature = ['street', 'postcode', 'outcode', 'latitude', 'isFreehold', 'have_area', \
              'longitude', 'bathrooms', 'bedrooms', 'rooms', 'floorAreaSqM', 'livingRooms', \
              'tenure', 'propertyType', 'currentEnergyRating', 'sale_year', 'sale_month']

X_train = train_df.drop('price', axis = 1)
y_train = train_df['price']

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", test_df.shape)

# **5. Modeling**

In [ ]:
scaler = StandardScaler()
X_train[X1_feature] = scaler.fit_transform(X_train[X1_feature])
test_df[X1_feature] = scaler.transform(test_df[X1_feature])

In [ ]:
class HybridModel(BaseEstimator, RegressorMixin):
    def __init__(self, trend_model, machine_model, trend_cols, machine_cols, all_columns):
        self.trend_model = trend_model
        self.machine_model = machine_model
        self.trend_cols = trend_cols
        self.machine_cols = machine_cols
        self.all_columns = all_columns
        self.ec = None

    def fit(self, X, y):
        y = np.log1p(y) 
        
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.all_columns)
        
        X1 = X[self.trend_cols]
        X2 = X[self.machine_cols]
        self.ec = Encoder()
        X2 = self.ec.fit_transform(X2, y)
        
        self.trend_model.fit(X1, y)
        trend_pred = self.trend_model.predict(X1)
        residual = y - trend_pred
        self.machine_model.fit(X2, residual)
        return self

    def predict(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.all_columns)
            
        X1 = X[self.trend_cols]
        X2 = X[self.machine_cols]
        X2 = self.ec.transform(X2)
        
        trend_pred = self.trend_model.predict(X1)
        machine_pred = self.machine_model.predict(X2)
        return np.expm1(trend_pred + machine_pred)

In [ ]:
model = {
    'HybridModel': Pipeline([
        ('Model', HybridModel(
            trend_model = ElasticNet(),
            machine_model = XGBRegressor(),
            trend_cols = X1_feature,
            machine_cols = X2_feature,
            all_columns = X_train.columns
        ))
    ]),
}

# **6. Grid search**

In [ ]:
params_grid = {
    'HybridModel': {
        'Model__trend_model__alpha': [0.008, 0.01, 0.005],
        'Model__machine_model__n_estimators': [500, 550, 600],
        'Model__machine_model__max_depth': [9, 10, 11],
        'Model__machine_model__learning_rate': [0.008, 0.1, 0.5],
    }
}

In [ ]:
best_model = {}
for name, pipeline in model.items():
  grid_search = GridSearchCV(pipeline, params_grid[name], cv=5, scoring='neg_mean_absolute_error', n_jobs = -1, verbose = 2, error_score='raise')
  grid_search.fit(X_train, y_train)
  print(name, ':', grid_search.best_params_, ' mae:', -grid_search.best_score_)
  best_model[name] = grid_search.best_estimator_

In [ ]:
grid_search.cv_results_

# **7. Chosing models**

In [ ]:
model_chosens = [
    ('HybridModel', best_model['HybridModel']),
]

In [ ]:
combine_model = VotingRegressor(estimators = model_chosens)

# **8. Retrain all**

In [ ]:
combine_model.fit(X_train, y_train)

# **9. Build submission**

In [ ]:
submission = pd.read_csv('/kaggle/input/london-house-price-prediction-advanced-techniques/sample_submission.csv')
submission['price'] = combine_model.predict(test_df.drop('price', axis = 1))
submission.to_csv('submission.csv', index=False)